# The config: what runs where, and how to read it

An `LRPConfig` says exactly which rule runs at every node of the autograd graph. This notebook builds two small models, walks through the vocabulary, shows the errors you get for anything that is not right, and reads back what ran with `explain`.

Nothing here needs a download; every cell runs on CPU in seconds.

In [1]:
import sys
sys.path.insert(0, '..')          # examples/showcase: _common
sys.path.insert(0, '../../..')    # repo root: autoLRP (or `pip install -e .`)
import torch
import torch.nn as nn
import torch.nn.functional as F

import autolrp
from autolrp import LRPConfig, BASE, CPLRP, ATTNLRP, explain, explain_summary
torch.manual_seed(0)

## Two models

A bias-free CNN, on which the relevance sum must equal the output relevance exactly (nothing absorbs), and a two-layer transformer block with the parts every language model has: layer norm, attention, a residual add, a GELU MLP.

In [2]:
cnn = nn.Sequential(
    nn.Conv2d(3, 8, 3, padding=1, bias=False), nn.ReLU(),
    nn.Conv2d(8, 16, 3, padding=1, bias=False), nn.ReLU(),
    nn.AdaptiveAvgPool2d(1), nn.Flatten(),
    nn.Linear(16, 10, bias=False),
).eval()
img = torch.randn(1, 3, 32, 32)

class Block(nn.Module):
    def __init__(self, d=32, h=4):
        super().__init__()
        self.h = h
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.qkv, self.o = nn.Linear(d, 3 * d), nn.Linear(d, d)
        self.ff1, self.ff2 = nn.Linear(d, 4 * d), nn.Linear(4 * d, d)
    def forward(self, x):
        B, T, D = x.shape
        q, k, v = self.qkv(self.ln1(x)).chunk(3, -1)
        split = lambda t: t.view(B, T, self.h, D // self.h).transpose(1, 2)
        a = F.scaled_dot_product_attention(split(q), split(k), split(v), is_causal=True)
        x = x + self.o(a.transpose(1, 2).reshape(B, T, D))
        return x + self.ff2(F.gelu(self.ff1(self.ln2(x))))

transformer = nn.Sequential(Block(), Block(), nn.LayerNorm(32), nn.Linear(32, 5)).eval()
tokens = torch.randn(1, 6, 32)

## The base table

`BASE` has one entry per rule-bearing node name (the autograd node name without its version digit: linear layers, products, adds, softmax, layer norm, activations, reductions), each the default of that node's rule table, plus one entry keyed by a fact (a label an analyzer attaches to a node; here, the normalization statistic in a mul, div or sub). `LRPConfig()` is `BASE` as it is.

In [3]:
for k, v in BASE.items():
    print(f'{k:22s} {v}')

AddmmBackward          epsilon
MmBackward             epsilon
BmmBackward            epsilon
ConvolutionBackward    epsilon
MulBackward            proportional
DivBackward            proportional
AddBackward            proportional
SubBackward            proportional
LogSoftmaxBackward     passthrough
SoftmaxBackward        passthrough
NativeLayerNormBackward identity
LayerNormBackward      identity
ReluBackward           passthrough
LeakyReluBackward      passthrough
GeluBackward           passthrough
SiluBackward           passthrough
TanhBackward           passthrough
SigmoidBackward        passthrough
HardtanhBackward       passthrough
HardswishBackward      passthrough
HardsigmoidBackward    passthrough
EluBackward            passthrough
SeluBackward           passthrough
CeluBackward           passthrough
SoftplusBackward       passthrough
SoftsignBackward       passthrough
LogSigmoidBackward     passthrough
MishBackward           passthrough
ExpBackward            passthrough
Lo

In [4]:
x = autolrp.tensor(img)
out = cnn(x)
out[0, 3].lrp()                                  # BASE
print('CNN, no biases:  sum R =', round(float(x.relevance.sum()), 6), '  (1.0 left the output)')

CNN, no biases:  sum R = 0.999999   (1.0 left the output)


## Overriding entries

Write the dict you mean, starting from `BASE`. The presets do the same thing in one call; `composite()` also sets `eps=1e-6`, the stabilizer the recipe was published with (the `BASE` default is `1e-11`, which is LRP-0 in practice).

In [5]:
configs = {
    'BASE':                    LRPConfig(),
    'z+ on conv':              LRPConfig(rule={**BASE, 'ConvolutionBackward': 'zplus'}, eps=1e-6),
    'gamma 0.25 on conv':      LRPConfig(rule={**BASE, 'ConvolutionBackward': ('gamma', {'gamma': 0.25})}),
    'LRPConfig.composite()':   LRPConfig.composite(),
}
maps = {}
for name, cfg in configs.items():
    x = autolrp.tensor(img.clone())
    cnn(x)[0, 3].lrp(config=cfg)
    maps[name] = x.relevance.clone()
    print(f'{name:24s} sum R = {float(x.relevance.sum()):+.4f}   conv entry = {cfg.rule["ConvolutionBackward"]}')

print()
print('z+ on conv == composite():', torch.equal(maps['z+ on conv'], maps['LRPConfig.composite()']), ' (composite() is that dict with eps=1e-6)')
print('BASE != z+ on conv:       ', not torch.allclose(maps['BASE'], maps['z+ on conv']))

BASE                     sum R = +1.0000   conv entry = epsilon
z+ on conv               sum R = +1.0000   conv entry = zplus
gamma 0.25 on conv       sum R = +1.0000   conv entry = ('gamma', {'gamma': 0.25})
LRPConfig.composite()    sum R = +1.0000   conv entry = zplus

z+ on conv == composite(): True  (composite() is that dict with eps=1e-6)
BASE != z+ on conv:        True


## Everything else is an error

A key that is not a node name or a registered fact, a rule the key's table cannot run, a key with a version digit, `'default'`, an alias, `'detach'` without a fact, and a partial dict that leaves a node of the graph without an entry. The first six fail at construction; the last one at install, naming the node and the key to add.

In [6]:
attempts = {
    "rule='epsilon'":                             lambda: LRPConfig(rule='epsilon'),
    "{**BASE, 'default': 'epsilon'}":             lambda: LRPConfig(rule={**BASE, 'default': 'epsilon'}),
    "{**BASE, 'linear': 'zplus'}":                lambda: LRPConfig(rule={**BASE, 'linear': 'zplus'}),
    "{**BASE, 'MulBackward0': 'proportional'}":   lambda: LRPConfig(rule={**BASE, 'MulBackward0': 'proportional'}),
    "{**BASE, 'MulBackward': 'zbox'}":            lambda: LRPConfig(rule={**BASE, 'MulBackward': 'zbox'}),
    "{**BASE, 'BmmBackward': 'detach'}":          lambda: LRPConfig(rule={**BASE, 'BmmBackward': 'detach'}),
    "{'BmmBackward': 'epsilon'} on the CNN":      lambda: cnn(autolrp.tensor(img.clone()))[0, 3].lrp(config=LRPConfig(rule={'BmmBackward': 'epsilon'})),
}
for label, f in attempts.items():
    try:
        f()
        print(f'{label:44s} -> ran')
    except (ValueError, TypeError) as e:
        print(f'{label:44s} -> {type(e).__name__}: {str(e)[:80]}')

rule='epsilon'                               -> TypeError: rule must be a dict keyed by node name or fact name; start from autolrp.BASE, e.
{**BASE, 'default': 'epsilon'}               -> ValueError: unknown rule key 'default': not a node name ['AbsBackward', 'AddBackward', 'Addm
{**BASE, 'linear': 'zplus'}                  -> ValueError: unknown rule key 'linear': not a node name ['AbsBackward', 'AddBackward', 'Addmm
{**BASE, 'MulBackward0': 'proportional'}     -> ValueError: rule key 'MulBackward0' carries a version digit; write 'MulBackward'
{**BASE, 'MulBackward': 'zbox'}              -> ValueError: rule entry 'MulBackward'='zbox': 'zbox' is not a choice here. Choices: ['proport
{**BASE, 'BmmBackward': 'detach'}            -> ValueError: rule entry 'BmmBackward'='detach': 'detach' needs by=<fact name>
{'BmmBackward': 'epsilon'} on the CNN        -> ValueError: no entry for node 'MmBackward0'. Add 'MmBackward' (or a fact the node carries) t


## Reading back what ran

`explain` installs the hooks, records which entry addressed each node and which function it picked, and removes the hooks again; no backward runs. `explain_summary` prints one line per distinct combination. Under the `CPLRP` fragment both products of every attention carry the `bilinear` fact and resolve through its entry, `('detach', {'by': 'attention_weights'})`. The `A @ V` product carries `attention_weights` as well, so the epsilon rule attributes the values only; the `Q @ K^T` product does not, so the table's default runs there: epsilon on both operands, half the relevance each.

In [7]:
x = autolrp.tensor(tokens)
rows = explain(transformer(x)[0, -1, 2], LRPConfig(rule={**BASE, **CPLRP}))
print(explain_summary([r for r in rows if r[2] != 'native gradient']))

count  node                      key                   what
    9  AddmmBackward0            AddmmBackward         epsilon (lhs)
    6  AddBackward               AddBackward           proportional
    5  NativeLayerNormBackward0  NativeLayerNormBackward  layernorm_identity
    2  BmmBackward0              bilinear              epsilon (rhs)
    2  BmmBackward0              bilinear              epsilon (both)
    2  GeluBackward              GeluBackward          passthrough
    2  MulBackward0              MulBackward           proportional (lhs)
    2  SoftmaxBackward           SoftmaxBackward       passthrough


In [8]:
rows = explain(transformer(autolrp.tensor(tokens))[0, -1, 2], LRPConfig.attnlrp())
print(explain_summary([r for r in rows if r[2] != 'native gradient']))

count  node                      key                   what
    9  AddmmBackward0            AddmmBackward         epsilon (lhs)
    6  AddBackward               AddBackward           proportional
    5  NativeLayerNormBackward0  NativeLayerNormBackward  layernorm_identity
    4  BmmBackward0              BmmBackward           epsilon (both)
    2  GeluBackward              GeluBackward          elementwise_yx
    2  MulBackward0              MulBackward           proportional (lhs)
    2  SoftmaxBackward           SoftmaxBackward       softmax_jacobian


`native gradient` rows are shape ops (views, transposes, selects), where the gradient already routes relevance to the right place. They are filtered out above; drop the filter to see them.

## Three conventions

1. Relevance flows to what you wrapped with `autolrp.tensor`. Every other tensor, a parameter, a constant, a plain tensor with `requires_grad`, is a weight. A frozen model (`requires_grad=False` everywhere) gives the same map as a trainable one.
2. A constant that multiplies, divides or negates passes relevance through unchanged (the `1/sqrt(d)` inside attention, a rotary embedding, a fixed mask).
3. A constant that is added is a bias, and a bias keeps its share; a layer with a bias emits less than it receives.

In [9]:
X = tokens.clone()
x1 = autolrp.tensor(X.clone()); transformer(x1)[0, -1, 2].lrp(config=LRPConfig(rule={**BASE, **ATTNLRP}))
for p in transformer.parameters():
    p.requires_grad_(False)
x2 = autolrp.tensor(X.clone()); transformer(x2)[0, -1, 2].lrp(config=LRPConfig(rule={**BASE, **ATTNLRP}))
for p in transformer.parameters():
    p.requires_grad_(True)
print('frozen == trainable:', torch.equal(x1.relevance, x2.relevance))

x = autolrp.tensor(torch.tensor([1.0, -2.0, 3.0])); (x * 0.25).sum().lrp()
print('x * 0.25 then sum:  sum R =', float(x.relevance.sum()))
x = autolrp.tensor(torch.tensor([1.0, -2.0, 3.0])); (x + torch.tensor([2.0, 2.0, 2.0])).sum().lrp()
print('x + [2, 2, 2] then sum: sum R =', round(float(x.relevance.sum()), 3), ' (the constant is a bias and kept its share)')

frozen == trainable: True
x * 0.25 then sum:  sum R = 1.0
x + [2, 2, 2] then sum: sum R = 0.5  (the constant is a bias and kept its share)


## Two inputs

A second input is a weight until you wrap it too. With both wrapped, a product of the two is bilinear and the relevance is split between them.

In [10]:
a = autolrp.tensor(torch.randn(2, 3)); w = torch.randn(3, 4, requires_grad=True)
(a @ w).sum().lrp()
print('unwrapped second operand: R to a =', round(float(a.relevance.sum()), 4))

a = autolrp.tensor(torch.randn(2, 3)); b = autolrp.tensor(torch.randn(3, 4))
(a @ b).sum().lrp()
print('both wrapped:             R to a =', round(float(a.relevance.sum()), 4), ' R to b =', round(float(b.relevance.sum()), 4))

unwrapped second operand: R to a = 1.0
both wrapped:             R to a = 0.5  R to b = 0.5
